In [1]:
import os
import json
import math
from typing import List, Dict, Any, Tuple
from neo4j import GraphDatabase, basic_auth

# If you use LangChain's ChatOpenAI (as shown in your snippet)
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
# OpenAI embeddings (official SDK)
import openai
import numpy as np
# OpenAI v1 SDK
from openai import OpenAI
import numpy as np

In [2]:
# # -----------------------------
# # Config / setup
# # -----------------------------
# URI = os.environ.get("NEO4J_URI", "neo4j+s://62b9e173.databases.neo4j.io")  # Aura example
# USER = os.environ.get("NEO4J_USERNAME", "neo4j")
# PASSWORD = os.environ.get("NEO4J_PASSWORD", "Thesis*1234")
# DB = os.getenv("NEO4J_DATABASE", "neo4j")

# OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# # LLM for topic extraction
# llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)

# # Embedding model + client
# EMBED_MODEL = "text-embedding-3-small"
# oa_client = OpenAI(api_key=OPENAI_API_KEY)

# # Neo4j driver
# driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))


In [3]:

# def verify():
#     print("[INFO] Connecting to:", URI)
#     print("[INFO] DB:", DB)
#     print("[INFO] USER:", USER)
#     driver.verify_connectivity()
#     print("[INFO] Connected. Server info:", driver.get_server_info())


# # -----------------------------
# # 1) Extract the main topic from paragraph
# # -----------------------------
# TOPIC_SYS_PROMPT = """You extract the single most important keyword/topic that the paragraph is about.
# Respond with ONLY the topic text, no punctuation or quotes. Keep it concise (max 8 words).
# Examples:
# - "The fourth season of Chicago Fire..." -> Chicago Fire Season 4
# - "Apple unveiled the iPhone 16 Pro..." -> iPhone 16 Pro
# """

# def extract_main_topic(llm: ChatOpenAI, paragraph: str) -> str:
#     resp = llm.invoke([
#         SystemMessage(content=TOPIC_SYS_PROMPT),
#         HumanMessage(content=paragraph)
#     ])
#     topic = (resp.content or "").strip()
#     topic = topic.replace('"', '').replace("'", "").strip()
#     return topic


# # -----------------------------
# # 2) Embeddings (OpenAI v1)
# # -----------------------------
# def embed_texts(texts: List[str]) -> np.ndarray:
#     """
#     Returns normalized embeddings (N, D). Empty -> shape (0, D).
#     """
#     if not texts:
#         return np.zeros((0, 1536), dtype=np.float32)

#     # Call embeddings
#     emb = oa_client.embeddings.create(model=EMBED_MODEL, input=texts)
#     vectors = [d.embedding for d in emb.data]

#     arr = np.array(vectors, dtype=np.float32)
#     norms = np.linalg.norm(arr, axis=1, keepdims=True)
#     norms[norms == 0] = 1.0
#     return arr / norms


# def cosine_sim_matrix(vec_query: np.ndarray, vecs: np.ndarray) -> np.ndarray:
#     """
#     vec_query: (D,)
#     vecs: (N, D)
#     returns: (N,)
#     """
#     return vecs @ vec_query


# # -----------------------------
# # 3) Pull candidate node "names" from Neo4j
# # -----------------------------
# # We try to derive a display name:
# #   Prefer (n)-[:HAS_NAME]->(v:value_text) and use v.id
# #   Else use n.id
# #   If still missing, fall back to elementId(n)
# GET_NODE_NAMES_CYPHER = """
# MATCH (n)
# OPTIONAL MATCH (n)-[:HAS_NAME]->(v:value_text)
# WITH n, v
# RETURN
#   coalesce(n.id, elementId(n)) AS id,
#   coalesce(v.id, n.id, elementId(n)) AS name_id_like,
#   labels(n) AS graph_labels
# """

# def prettify_name(name_id_like: str) -> str:
#     if not name_id_like:
#         return ""
#     x = name_id_like
#     if x.startswith("val_"):
#         x = x[4:]
#     x = x.replace("_", " ")
#     return x.strip()

# def fetch_all_nodes_with_names() -> List[Dict[str, Any]]:
#     with driver.session(database=DB) as session:
#         rows = session.run(GET_NODE_NAMES_CYPHER).data()

#     results = []
#     for r in rows:
#         node_id = r.get("id")
#         raw_name = r.get("name_id_like") or node_id
#         pretty = prettify_name(raw_name)
#         graph_labels = r.get("graph_labels", [])
#         results.append({
#             "id": node_id,
#             "raw_name": raw_name,
#             "name": pretty if pretty else (node_id or ""),
#             "graph_labels": graph_labels,
#         })
#     return results


# # -----------------------------
# # 4) Find most similar node to topic
# # -----------------------------
# def find_most_similar_node(topic: str, candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
#     if not candidates:
#         return {}

#     texts = [c["name"] for c in candidates]
#     vecs = embed_texts(texts)           # (N, D)

#     q_vec = embed_texts([topic])[0]     # (D,)
#     sims = cosine_sim_matrix(q_vec, vecs)  # (N,)

#     best_idx = int(np.argmax(sims))
#     best = candidates[best_idx].copy()
#     best["similarity"] = float(sims[best_idx])
#     return best


# # -----------------------------
# # 5) Fetch all connections for that node
# # -----------------------------
# GET_NEIGHBORS_CYPHER = """
# MATCH (center {id: $id})
# OPTIONAL MATCH (center)-[r]-(nbr)
# RETURN
#   center.id AS center_id,
#   type(r)   AS rel_type,
#   CASE
#     WHEN r IS NULL THEN NULL
#     WHEN startNode(r) = center THEN 'OUT'
#     ELSE 'IN'
#   END AS direction,
#   coalesce(nbr.id, elementId(nbr)) AS neighbor_id,
#   labels(nbr) AS neighbor_graph_labels
# ORDER BY rel_type, neighbor_id
# """

# def fetch_connections(node_id: str) -> List[Dict[str, Any]]:
#     with driver.session(database=DB) as session:
#         rows = session.run(GET_NEIGHBORS_CYPHER, {"id": node_id}).data()
#     return [r for r in rows if r.get("rel_type") is not None]


# # -----------------------------
# # 6) Pretty-print results
# # -----------------------------
# def print_summary(topic: str, best_node: Dict[str, Any], neighbors: List[Dict[str, Any]]):
#     print("\n=== Topic Extracted ===")
#     print(topic)

#     if not best_node:
#         print("\n[WARN] No candidate nodes found in graph.")
#         return

#     print("\n=== Best Match Node ===")
#     print(f"ID: {best_node['id']}")
#     print(f"Name: {best_node['name']}")
#     print(f"Similarity: {best_node['similarity']:.4f}")
#     print(f"Labels (graph): {best_node.get('graph_labels')}")

#     print("\n=== Connections (all neighbors) ===")
#     if not neighbors:
#         print("[INFO] No connections found.")
#         return

#     for r in neighbors:
#         if r["direction"] == "IN":
#             print(f"{r['neighbor_id']} -[{r['rel_type']}]-> {best_node['id']}")
#         else:
#             print(f"{best_node['id']} -[{r['rel_type']}]-> {r['neighbor_id']}")


In [4]:
verify()

# User input paragraph
paragraph = (
   "Famous for his velvety voice, American singer Frank Sinatra surprisingly made a rendition of the song 'I Can't Help Falling in Love with You', setting a classic benchmark."          )
# 1) Extract main topic
topic = extract_main_topic(llm, paragraph)
print(f"[INFO] Extracted topic: {topic}")

# 2) Candidates from graph
candidates = fetch_all_nodes_with_names()
if not candidates:
    print("[WARN] Your graph seems empty (no nodes returned).")

# 3) Most similar node
best = find_most_similar_node(topic, candidates)
if not best:
    print("[WARN] Could not determine the most similar node.")


# 4) Fetch all connections
neighbors = fetch_connections(best["id"])

# 5) Print result
print_summary(topic, best, neighbors)

NameError: name 'verify' is not defined

In [ ]:
# for i in neighbors:

#     print(i)

{'center_id': 'frank', 'rel_type': 'MARRIES', 'direction': 'IN', 'neighbor_id': 'julie', 'neighbor_graph_labels': ['person']}


In [5]:
# main.py
import os
import json
from typing import List, Dict, Any
import numpy as np
from neo4j import GraphDatabase, basic_auth

# --- LLMs ---
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from openai import OpenAI

# =========================
# ENV / CONFIG
# =========================
URI = os.environ.get("NEO4J_URI", "neo4j://localhost:7687")          # change if Aura
USER = os.environ.get("NEO4J_USERNAME", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD", "password")
DB = os.environ.get("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # must be set
EMBED_MODEL = "text-embedding-3-small"         # or text-embedding-3-large

# =========================
# OPENAI / NEO4J CLIENTS
# =========================
llm = ChatOpenAI(temperature=0, model_name="gpt-4o", api_key=OPENAI_API_KEY)
oa_client = OpenAI(api_key=OPENAI_API_KEY)
driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))

def verify():
    print("[INFO] Connecting to:", URI)
    print("[INFO] DB:", DB)
    print("[INFO] USER:", USER)
    driver.verify_connectivity()
    print("[INFO] Connected. Server info:", driver.get_server_info())

# =========================
# LLM: TOPIC EXTRACTION
# =========================
TOPIC_SYS_PROMPT = """
You are given a user query/paragraph. Your job is to output a SHORT TOPIC STRING
that names the concrete subject being asked about — NOT the answer.

Rules:
- Never answer the question.
- Return the entity/work/concept mentioned in the query that the question is about.
- For WH-questions (who/when/where/which/how many, etc.), point to the referenced thing,
  not the unknown variable. E.g., “Who recorded X?” → return X (the song), not the artist.
- Prefer the canonical name/title; Title Case; no quotes or punctuation; max 8 words.
- If disambiguation helps, add a type in parentheses (song/film/book/series), e.g. "Chicago Fire Season 4(series)".
- Output ONLY the topic string.

Sample Examples:

Q: When was the Eiffel Tower built?
→ Eiffel Tower

Q: Who wrote The Great Gatsby?
→ The Great Gatsby

Q: Compare iPhone 15 and Galaxy S23 cameras.
→ iPhone 15 vs Galaxy S23 cameras

Q: What is Elon Musk's net worth?
→ Elon Musk

Q: Capital city of France?
→ France capital

"""

def extract_main_topic(llm: ChatOpenAI, paragraph: str) -> str:
    resp = llm.invoke([
        SystemMessage(content=TOPIC_SYS_PROMPT),
        HumanMessage(content=paragraph)
    ])
    topic = (resp.content or "").strip()
    return topic.replace('"', '').replace("'", "").strip()

# =========================
# EMBEDDINGS (BATCHED & SAFE)
# =========================
BATCH_SIZE = 256
MAX_NAME_CHARS = 200
MAX_CANDIDATES = 20000
PREFILTER_MIN_MATCH = 1  # # topic tokens required in name

def _sanitize_text(s: str) -> str:
    if not s: return " "
    s = s.strip()
    return s[:MAX_NAME_CHARS] if len(s) > MAX_NAME_CHARS else s

def _embed_batch(oa_client: OpenAI, texts: List[str]) -> np.ndarray:
    texts = [_sanitize_text(t) for t in texts]
    resp = oa_client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return vecs / norms

def embed_texts_batched(oa_client: OpenAI, texts: List[str], batch_size: int = BATCH_SIZE) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1536), dtype=np.float32)
    out = []
    for i in range(0, len(texts), batch_size):
        out.append(_embed_batch(oa_client, texts[i:i+batch_size]))
    return np.vstack(out)

def cosine_sim_matrix(vec_query: np.ndarray, vecs: np.ndarray) -> np.ndarray:
    return vecs @ vec_query

# =========================
# CANDIDATE FETCH (with cheap prefilter)
# =========================
def prettify_name(name_id_like: str) -> str:
    if not name_id_like:
        return ""
    x = name_id_like
    if x.startswith("val_"):
        x = x[4:]
    return x.replace("_", " ").strip()

# Generic all-nodes fallback (kept for safety)
GET_NODE_NAMES_CYPHER_ALL = """
MATCH (n)
OPTIONAL MATCH (n)-[:HAS_NAME]->(v:value_text)
WITH n, v
RETURN
  coalesce(n.id, elementId(n)) AS id,
  coalesce(v.id, n.id, elementId(n)) AS name_id_like,
  labels(n) AS graph_labels
"""

def find_top_k_similar_nodes(oa_client: OpenAI, topic: str, candidates: List[Dict[str, Any]], k: int = 3) -> List[Dict[str, Any]]:
    if not candidates:
        return []
    pool = _cheap_prefilter_candidates_py(topic, candidates)
    if len(pool) > MAX_CANDIDATES:
        pool = pool[:MAX_CANDIDATES]

    # Embed query once
    q_vec = _embed_batch(oa_client, [topic])[0]  # shape (D,)

    names = [c["name"] for c in pool]
    sims = []

    # Batch-embed candidate names and compute sims
    for i in range(0, len(names), BATCH_SIZE):
        vecs = _embed_batch(oa_client, names[i:i+BATCH_SIZE])  # shape (B, D), already L2-normalized
        # since q_vec is normalized too, dot = cosine
        sims_chunk = vecs @ q_vec
        sims.extend(sims_chunk.tolist())

    # top-k indices
    if not sims:
        return []
    k = min(k, len(sims))
    top_idx = np.argpartition(sims, -k)[-k:]              # unsorted top-k
    top_idx = top_idx[np.argsort(np.array(sims)[top_idx])][::-1]  # sort desc

    results = []
    for idx in top_idx:
        item = pool[idx].copy()
        item["similarity"] = float(sims[idx])
        results.append(item)
    return results


# Cheap token prefilter directly in Cypher (case-insensitive CONTAINS on id/name)
def fetch_nodes_prefiltered(topic: str) -> List[Dict[str, Any]]:
    tokens = [t for t in topic.lower().replace("-", " ").split() if len(t) >= 3]
    if not tokens:
        # No useful tokens → return all nodes (beware size)
        with driver.session(database=DB) as session:
            rows = session.run(GET_NODE_NAMES_CYPHER_ALL).data()
    else:
        # Build a WHERE with OR of tokens against id and name_id_like
        # We’ll MATCH all, but filter in WHERE
        where_clauses = []
        params = {}
        for idx, tok in enumerate(tokens):
            p = f"tok{idx}"
            params[p] = tok
            where_clauses.append(f"toLower(coalesce(v.id, n.id, elementId(n))) CONTAINS ${p}")
            where_clauses.append(f"toLower(coalesce(n.id, elementId(n))) CONTAINS ${p}")
        where = " OR ".join(where_clauses)
        cypher = f"""
        MATCH (n)
        OPTIONAL MATCH (n)-[:HAS_NAME]->(v:value_text)
        WITH n, v
        WHERE {where}
        RETURN
          coalesce(n.id, elementId(n)) AS id,
          coalesce(v.id, n.id, elementId(n)) AS name_id_like,
          labels(n) AS graph_labels
        LIMIT {MAX_CANDIDATES}
        """
        with driver.session(database=DB) as session:
            rows = session.run(cypher, params).data()

    results = []
    for r in rows:
        node_id = r.get("id")
        raw_name = r.get("name_id_like") or node_id
        results.append({
            "id": node_id,
            "raw_name": raw_name,
            "name": prettify_name(raw_name) or (node_id or ""),
            "graph_labels": r.get("graph_labels", []),
        })
    return results

def _cheap_prefilter_candidates_py(topic: str, candidates: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if not candidates or not topic:
        return candidates
    topic_tokens = [t for t in topic.lower().replace("-", " ").split() if len(t) >= 3]
    if not topic_tokens:
        return candidates
    kept = []
    for c in candidates:
        name = (c.get("name") or "").lower()
        hits = sum(1 for t in topic_tokens if t in name)
        if hits >= PREFILTER_MIN_MATCH:
            kept.append(c)
    return kept if len(kept) >= 50 else candidates

def find_most_similar_node(oa_client: OpenAI, topic: str, candidates: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not candidates:
        return {}
    # Optional Python prefilter as a second pass
    pool = _cheap_prefilter_candidates_py(topic, candidates)
    if len(pool) > MAX_CANDIDATES:
        pool = pool[:MAX_CANDIDATES]

    # Embed query once
    q_vec = _embed_batch(oa_client, [topic])[0]  # (D,)

    best_idx = -1
    best_sim = -1.0
    names = [c["name"] for c in pool]

    for i in range(0, len(names), BATCH_SIZE):
        vecs = _embed_batch(oa_client, names[i:i+BATCH_SIZE])  # (B, D)
        sims = vecs @ q_vec
        j = int(np.argmax(sims))
        if sims[j] > best_sim:
            best_sim = float(sims[j])
            best_idx = i + j

    if best_idx >= 0:
        best = pool[best_idx].copy()
        best["similarity"] = best_sim
        return best
    return {}

# =========================
# NEIGHBORS (with fallback to elementId)
# =========================
GET_NEIGHBORS_BY_ID = """
MATCH (center {id: $id})
OPTIONAL MATCH (center)-[r]-(nbr)
RETURN
  center.id AS center_id,
  type(r)   AS rel_type,
  CASE
    WHEN r IS NULL THEN NULL
    WHEN startNode(r) = center THEN 'OUT'
    ELSE 'IN'
  END AS direction,
  coalesce(nbr.id, elementId(nbr)) AS neighbor_id,
  labels(nbr) AS neighbor_graph_labels
ORDER BY rel_type, neighbor_id
"""

GET_NEIGHBORS_BY_COALESCE = """
MATCH (center)
WHERE coalesce(center.id, elementId(center)) = $id
OPTIONAL MATCH (center)-[r]-(nbr)
RETURN
  coalesce(center.id, elementId(center)) AS center_id,
  type(r)   AS rel_type,
  CASE
    WHEN r IS NULL THEN NULL
    WHEN startNode(r) = center THEN 'OUT'
    ELSE 'IN'
  END AS direction,
  coalesce(nbr.id, elementId(nbr)) AS neighbor_id,
  labels(nbr) AS neighbor_graph_labels
ORDER BY rel_type, neighbor_id
"""

def fetch_connections(node_id: str) -> List[Dict[str, Any]]:
    with driver.session(database=DB) as session:
        rows = session.run(GET_NEIGHBORS_BY_ID, {"id": node_id}).data()
    rows = [r for r in rows if r.get("rel_type") is not None]
    if rows:
        return rows
    # fallback if node had no `id` property and we were given elementId
    with driver.session(database=DB) as session:
        rows2 = session.run(GET_NEIGHBORS_BY_COALESCE, {"id": node_id}).data()
    return [r for r in rows2 if r.get("rel_type") is not None]

# =========================
# MAIN (demo run)
# =========================

verify()

paragraph = "The fourth season of the popular drama series, Chicago Fire, contains a total of 24 episodes. This season continued to engage viewers with thrilling and dramatic moments."
topic = extract_main_topic(llm, paragraph)
print(f"[INFO] Extracted topic: {topic}")

# Pull candidates with Cypher prefilter to avoid embedding the whole graph
candidates = fetch_nodes_prefiltered(topic)
if not candidates:
    print("[WARN] Your graph seems empty (no nodes returned).")
    exit(0)

top3 = find_top_k_similar_nodes(oa_client, topic, candidates, k=3)
if not top3:
    print("[WARN] Could not determine similar nodes.")
    exit(0)

# Print top-3 and their connections
print("\n=== Top 3 Matching Nodes ===")
for rank, node in enumerate(top3, 1):
    print(f"\n-- #{rank} --")
    print(f"ID: {node['id']}")
    print(f"Name: {node['name']}")
    print(f"Similarity: {node['similarity']:.4f}")
    print(f"Labels (graph): {node.get('graph_labels')}")
    neighbors = fetch_connections(node["id"])
    if not neighbors:
        print("[INFO] No connections found.")
    else:
        print("Connections:")
        for r in neighbors:
            if r["direction"] == "IN":
                print(f"  {r['neighbor_id']} -[{r['rel_type']}]-> {node['id']}")
            else:
                print(f"  {node['id']} -[{r['rel_type']}]-> {r['neighbor_id']}")


[INFO] Connecting to: neo4j+s://62b9e173.databases.neo4j.io
[INFO] DB: neo4j
[INFO] USER: neo4j
[INFO] Connected. Server info: <neo4j.api.ServerInfo object at 0x7343680134c0>
[INFO] Extracted topic: Chicago Fire Season 4 (series)

=== Top 3 Matching Nodes ===

-- #1 --
ID: chicago_fire_season_4
Name: chicago fire season 4
Similarity: 0.8542
Labels (graph): ['work']
Connections:
  chicago_fire_season_4 -[CONCLUDED_ON]-> val_2016_05_17
  chicago_fire_season_4 -[EPISODE_COUNT]-> val_23
  chicago_fire_season_4 -[EXECUTIVE_PRODUCED_BY]-> dick_wolf
  chicago_fire_season_4 -[FEATURES]-> ambulance_61
  chicago_fire_season_4 -[FEATURES]-> battalion_25
  chicago_fire_season_4 -[FEATURES]-> engine_51
  chicago_fire_season_4 -[FEATURES]-> squad_3
  chicago_fire_season_4 -[FEATURES]-> truck_81
  chicago_fire_season_4 -[LOCATED_AT]-> chicago_fire_department
  chicago_fire_season_4 -[ORDERED_BY]-> nbc
  chicago_fire_season_4 -[ORDERED_ON]-> val_2015_02_05
  chicago_fire_season_4 -[PREMIERE_DATE]-> va

In [6]:
import json
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

# ---- 1) System prompt (only show mismatches) ----
SYS_PROMPT = """
You are a comparer. Your job is ONLY to check whether the factual statements in a paragraph match the provided knowledge-graph (KG) facts. If something does not match, list it. If everything matches, return an empty list.

INPUT (user message will provide this JSON):
{
  "paragraph": "<full text>",
  "kg_facts": [
    {
      "subject": "text name of subject (e.g., 'Chicago Fire Season 4')",
      "predicate": "ALL_CAPS_UNDERSCORE (e.g., 'PREMIERE_DATE', 'EPISODE_COUNT')",
      "object": "text for value or entity name (e.g., '2015-10-13' or '23' or 'Dick Wolf')",
      "object_type": "value_date | value_number | value_text | entity"
    }
  ],
  "center": "optional textual center/topic to anchor subjects, e.g., 'Chicago Fire Season 4'"
}

REQUIREMENTS:
- Split the paragraph into short factual clauses.
- From each clause, extract only clear facts about the center/topic if provided; otherwise general facts present.
- Normalize before comparing:
  • Dates → ISO (YYYY-MM-DD if available, else YYYY-MM or YYYY).
  • Numbers → compare numerically.
  • Text/entities → case/space/underscore insensitive (e.g., 'Season 4' ≈ 'season_4').
  • Predicate meaning → treat obvious paraphrases as equivalent (e.g., “premiered on” ≈ PREMIERE_DATE; “ended/concluded” ≈ CONCLUDED_ON).
- Compare each fact from the paragraph to the KG. If KG has a compatible fact with the same meaning and value, consider it MATCHED and do not include it in the output.
- If KG has the same fact with a different value, that is a CONTRADICTION and MUST be included.
- If KG has no relevant fact, DO NOT include it (we only report mismatches, not missing facts).

OUTPUT:
Return a single valid JSON object with ONLY the mismatches:

{
  "contradictions": [
    {
      "clause": "original clause text",
      "predicate": "normalized predicate (ALL_CAPS)",
      "paragraph_value": "value from paragraph (normalized)",
      "kg_value": "value from KG that conflicts (normalized)",
      "subject": "subject text you compared (ideally the center/topic)",
      "reason": "short reason for why they conflict"
    }
  ]
}

NOTES:
- If everything matches, return: { "contradictions": [] }
- No prose, no explanations beyond the JSON fields above.
"""

# ---- 2) Helper: build kg_facts from Neo4j neighbors rows ----
# neighbors rows look like:
# {'center_id': 'chicago_fire_season_4', 'rel_type': 'EPISODE_COUNT', 'direction': 'OUT',
#  'neighbor_id': 'val_23', 'neighbor_graph_labels': ['value_number']}

def _pretty(s: str) -> str:
    if not s: return ""
    s = s.strip()
    if s.startswith("val_"):
        s = s[4:]
    return s.replace("_", " ").strip()

def _infer_object_type(labels):
    labels = set(labels or [])
    if "value_date" in labels: return "value_date"
    if "value_number" in labels: return "value_number"
    if "value_text" in labels: return "value_text"
    return "entity"

def build_kg_facts_from_neighbors(neighbors, center_name: str) -> list:
    facts = []
    for r in neighbors:
        predicate = r["rel_type"]              # keep as-is; the LLM will normalize by meaning
        obj_type = _infer_object_type(r.get("neighbor_graph_labels"))
        obj_name = _pretty(r["neighbor_id"])
        facts.append({
            "subject": center_name,            # human-friendly subject text
            "predicate": predicate,
            "object": obj_name,                # human-friendly object text/value
            "object_type": obj_type
        })
    return facts

# ---- 3) Example: pass paragraph + KG JSON to the LLM and parse JSON output ----
def run_mismatch_check(paragraph: str, neighbors: list, center_name: str, OPENAI_API_KEY: str):
    llm_2 = ChatOpenAI(
        model_name="gpt-4o",
        temperature=0,
        openai_api_key=OPENAI_API_KEY,
        # force JSON output:
        model_kwargs={"response_format": {"type": "json_object"}}
    )

    kg_facts = build_kg_facts_from_neighbors(neighbors, center_name)

    payload = {
        "paragraph": paragraph,
        "kg_facts": kg_facts,
        "center": center_name
    }

    resp = llm_2.invoke([
        SystemMessage(content=SYS_PROMPT),
        HumanMessage(content=json.dumps(payload))
    ])

    result = json.loads(resp.content)  # guaranteed JSON by response_format
    return result



# ---- 4) NEW: run checker over the top-K candidates ----
def run_mismatch_check_for_candidates(paragraph: str, candidates: list, OPENAI_API_KEY: str):
    """
    candidates: list of dicts like {'id': ..., 'name': ..., 'similarity': ...}
    Returns a consolidated JSON object keyed by center.
    """
    centers = []
    for c in candidates:
        center_id = c.get("id")
        center_name = c.get("name") or str(center_id)
        similarity = float(c.get("similarity", 0.0))

        # fetch neighbors for this center
        nbrs = fetch_connections(center_id)  # uses earlier-defined function

        # run mismatch check
        res = run_mismatch_check(paragraph, nbrs, center_name, OPENAI_API_KEY)

        centers.append({
            "id": center_id,
            "name": center_name,
            "similarity": similarity,
            "contradictions": res.get("contradictions", [])
        })

    return { "centers": centers }

# ---- 5) Execute for your current variables: paragraph + top3 ----
multi_result = run_mismatch_check_for_candidates(paragraph, top3, OPENAI_API_KEY)
print(json.dumps(multi_result, indent=2))

{
  "centers": [
    {
      "id": "chicago_fire_season_4",
      "name": "chicago fire season 4",
      "similarity": 0.8542200326919556,
      "contradictions": [
        {
          "clause": "The fourth season of the popular drama series, Chicago Fire, contains a total of 24 episodes.",
          "predicate": "EPISODE_COUNT",
          "paragraph_value": "24",
          "kg_value": "23",
          "subject": "chicago fire season 4",
          "reason": "The paragraph states 24 episodes, but the KG states 23 episodes."
        }
      ]
    },
    {
      "id": "val_fourth_season",
      "name": "fourth season",
      "similarity": 0.6252380013465881,
      "contradictions": []
    },
    {
      "id": "fourth_season",
      "name": "fourth season",
      "similarity": 0.6252380013465881,
      "contradictions": []
    }
  ]
}
